
# NeoLume — Jaundice Risk Model Training (Colab)

Trains and compares **two** risk models for the "Layer 2 — On-Device AI"
stage of the NeoLume architecture:

1. **CNN (MobileNetV3-Small, transfer learning)** — matches the deck's
   diagram directly, exports to `.h5` and a quantized `.tflite` for
   on-device use.
2. **XGBoost on hand-crafted colour features** (CIE LAB / HSV stats) —
   matches the diagram's "Feature Extraction → Risk Model" alternative
   path, trains in seconds, and is far more explainable — useful as a
   sanity-check baseline and a fallback if the CNN overfits.

Both are evaluated the same way: **accuracy, precision, recall, F1,
AUC-ROC**, confusion matrix, ROC curve, PR curve, and a recall-first
threshold sweep (a missed jaundiced newborn is worse than a false alarm,
so this notebook optimizes for **recall**, not just accuracy).

**Before running:** Runtime → Change runtime type → **GPU** (T4 is enough).

**Dataset assumption:** class imbalance of roughly 200 "jaundice" images vs
1100 "normal" images (adjust `CLASS_DIRS` in the config cell if your
folder names differ from the Kaggle set referenced in the deck:
https://www.kaggle.com/datasets/aiolapo/jaundice-image-data).

**Honest scope note:** the diagram's "Region Detection" (YOLOv8n/MediaPipe
sclera/skin localization) and "Colour Calibration" (against a physical
card) happen on-device at capture time in the real app — this notebook's
public dataset has neither, so it trains on already-cropped images and
substitutes a gray-world auto white-balance as a calibration proxy. Flag
this gap to judges: the trained model should be re-validated once you have
calibration-card-normalized field images from your pilot.


## 1. Install & import dependencies

In [ ]:

!pip -q install xgboost albumentations scikit-learn seaborn --upgrade


In [ ]:

import os, json, shutil, random, pathlib, math
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_recall_curve, average_precision_score, f1_score,
    precision_score, recall_score, accuracy_score, roc_auc_score
)

import xgboost as xgb

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


## 2. Config

In [ ]:

# --- EDIT THESE FOR YOUR SETUP ---
DATA_ROOT = "/content/data"          # after download/extraction, expect DATA_ROOT/<class folders>
CLASS_DIRS = {
    "jaundice": 1,   # at-risk class -> label 1
    "normal":   0,   # label 0
}
IMG_SIZE = (224, 224)
BATCH_SIZE = 16
TEST_SPLIT = 0.15
VAL_SPLIT = 0.15          # of the remaining train+val portion
EPOCHS_HEAD = 10
EPOCHS_FINETUNE = 8
OUT_DIR = "/content/neolume_out"
os.makedirs(OUT_DIR, exist_ok=True)



## 3. Get the dataset

Two options — use whichever is easier for you:

**Option A — Kaggle API** (recommended, fastest)
1. Kaggle → Account → Create New API Token → downloads `kaggle.json`.
2. Run the cell below, upload `kaggle.json` when prompted.

**Option B — Google Drive**
Skip the Kaggle cell, instead mount Drive and point `DATA_ROOT` at your
uploaded dataset folder (uncomment the Drive cell).


In [ ]:

# --- Option A: Kaggle API download ---
from google.colab import files
print("Upload your kaggle.json (Kaggle -> Account -> Create New API Token)")
uploaded = files.upload()  # select kaggle.json

os.makedirs("/root/.kaggle", exist_ok=True)
shutil.move("kaggle.json", "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

!kaggle datasets download -d aiolapo/jaundice-image-data -p /content --unzip


In [ ]:

# --- Option B: Google Drive (uncomment to use instead of Kaggle) ---
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_ROOT = "/content/drive/MyDrive/neolume_data"


In [ ]:

# Inspect whatever got downloaded/extracted so we can fix CLASS_DIRS if
# the real folder names differ from our assumption.
def walk_preview(root, max_depth=3):
    root = Path(root)
    for p in sorted(root.rglob("*")):
        depth = len(p.relative_to(root).parts)
        if depth > max_depth:
            continue
        marker = "  " * (depth-1) + ("[DIR] " if p.is_dir() else "")
        if p.is_dir() or depth <= max_depth:
            print(marker + p.name)

# Try a couple of likely extraction locations
for candidate in ["/content", "/content/data", "/content/jaundice-image-data"]:
    if Path(candidate).exists():
        print(f"--- {candidate} ---")
        walk_preview(candidate, max_depth=2)
        print()


In [ ]:

# Once you've seen the real structure above, point DATA_ROOT at the folder
# that directly CONTAINS the class subfolders, and fix CLASS_DIRS names
# if needed, e.g.:
# DATA_ROOT = "/content/jaundice-image-data"
# CLASS_DIRS = {"Jaundice": 1, "Normal": 0}

assert Path(DATA_ROOT).exists(), (
    f"{DATA_ROOT} does not exist yet - update DATA_ROOT above to match "
    f"what the walk_preview() output showed, then re-run from this cell."
)


## 4. Build the file list + Exploratory Data Analysis

In [ ]:

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp"}

records = []
for folder_name, label in CLASS_DIRS.items():
    folder = Path(DATA_ROOT) / folder_name
    if not folder.exists():
        print(f"[warn] missing folder: {folder}")
        continue
    for f in folder.rglob("*"):
        if f.suffix.lower() in IMG_EXTS:
            records.append({"path": str(f), "label": label, "class_name": folder_name})

df = pd.DataFrame(records)
print(f"Total images found: {len(df)}")
print(df.groupby("class_name").size())
assert len(df) > 0, "No images found - check DATA_ROOT / CLASS_DIRS above."


In [ ]:

# Class balance bar chart
counts = df.groupby("class_name").size()
plt.figure(figsize=(5,4))
bars = plt.bar(counts.index, counts.values, color=["#B23A2E", "#1C8C82"])
plt.title("Class distribution")
plt.ylabel("Number of images")
for b in bars:
    plt.text(b.get_x()+b.get_width()/2, b.get_height()+5, str(int(b.get_height())),
              ha='center', fontsize=10)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/eda_class_balance.png", dpi=150)
plt.show()

imbalance_ratio = counts.max() / counts.min()
print(f"Imbalance ratio: {imbalance_ratio:.1f} : 1 — class weighting and/or "
      f"augmentation of the minority class is required (handled below).")


In [ ]:

# Sample image grid per class
fig, axes = plt.subplots(2, 6, figsize=(16, 6))
for row, (folder_name, label) in enumerate(CLASS_DIRS.items()):
    sample_paths = df[df["class_name"] == folder_name]["path"].sample(
        min(6, (df["class_name"]==folder_name).sum()), random_state=SEED
    ).tolist()
    for col in range(6):
        ax = axes[row, col]
        if col < len(sample_paths):
            img = cv2.cvtColor(cv2.imread(sample_paths[col]), cv2.COLOR_BGR2RGB)
            ax.imshow(img)
        ax.axis("off")
        if col == 0:
            ax.set_ylabel(folder_name, fontsize=12)
    axes[row, 0].text(-0.15, 0.5, folder_name, transform=axes[row,0].transAxes,
                       fontsize=13, va='center', ha='right', rotation=90)
plt.suptitle("Sample images per class")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/eda_sample_grid.png", dpi=150)
plt.show()


In [ ]:

# Image size distribution (sanity check for wildly inconsistent resolutions)
sizes = []
for p in df["path"].sample(min(200, len(df)), random_state=SEED):
    img = cv2.imread(p)
    if img is not None:
        sizes.append(img.shape[:2])
sizes = np.array(sizes)
plt.figure(figsize=(5,4))
plt.scatter(sizes[:,1], sizes[:,0], alpha=0.5, s=15)
plt.xlabel("width (px)"); plt.ylabel("height (px)")
plt.title("Sampled image dimensions")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/eda_image_sizes.png", dpi=150)
plt.show()


## 5. Stratified train / val / test split

In [ ]:

train_val_df, test_df = train_test_split(
    df, test_size=TEST_SPLIT, stratify=df["label"], random_state=SEED
)
train_df, val_df = train_test_split(
    train_val_df, test_size=VAL_SPLIT, stratify=train_val_df["label"], random_state=SEED
)

for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name}: {len(d)} images | class balance: "
          f"{d.groupby('class_name').size().to_dict()}")

split_summary = pd.DataFrame({
    "split": ["train","val","test"]*1,
}).assign(count=[len(train_df), len(val_df), len(test_df)])
plt.figure(figsize=(5,4))
plt.bar(["train","val","test"], [len(train_df), len(val_df), len(test_df)],
        color=["#1C8C82","#C98A2C","#B23A2E"])
plt.title("Split sizes")
plt.tight_layout()
plt.show()



## 6. Preprocessing & calibration proxy

Since this public dataset has no physical calibration card in-frame, we
approximate colour calibration with a **gray-world white balance** (assumes
the average scene colour should be neutral gray — a standard, well-known
proxy). In the real app, this step is replaced by the actual card-based
correction (see `feature_extraction.py`'s `calibrate_white_balance`).


In [ ]:

def gray_world_white_balance(img_bgr: np.ndarray) -> np.ndarray:
    img = img_bgr.astype(np.float32)
    mean_b, mean_g, mean_r = img[...,0].mean(), img[...,1].mean(), img[...,2].mean()
    mean_gray = (mean_b + mean_g + mean_r) / 3.0
    gains = np.array([mean_gray/max(mean_b,1e-3), mean_gray/max(mean_g,1e-3), mean_gray/max(mean_r,1e-3)])
    gains = np.clip(gains, 0.5, 2.0)
    out = np.clip(img * gains, 0, 255).astype(np.uint8)
    return out

def load_and_preprocess(path, size=IMG_SIZE, calibrate=True):
    img = cv2.imread(path)
    if img is None:
        return None
    if calibrate:
        img = gray_world_white_balance(img)
    img = cv2.resize(img, size)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img


In [ ]:

# Visual sanity check: before vs after calibration
sample_path = df["path"].iloc[0]
raw = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB)
calibrated = cv2.cvtColor(gray_world_white_balance(cv2.imread(sample_path)), cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1,2, figsize=(8,4))
axes[0].imshow(raw); axes[0].set_title("Raw"); axes[0].axis("off")
axes[1].imshow(calibrated); axes[1].set_title("After gray-world calibration"); axes[1].axis("off")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/calibration_before_after.png", dpi=150)
plt.show()


## 7. tf.data pipelines with augmentation (train only)

In [ ]:

augment_layer = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
    layers.RandomBrightness(0.15),
    layers.RandomContrast(0.15),
], name="augmentation")

def make_dataset(dframe, training: bool, batch_size=BATCH_SIZE):
    paths = dframe["path"].values
    labels = dframe["label"].values.astype("float32")

    def _load(path, label):
        path = path.numpy().decode("utf-8")
        img = load_and_preprocess(path, calibrate=True)
        if img is None:
            img = np.zeros(IMG_SIZE + (3,), dtype=np.uint8)
        return img.astype("float32"), label

    def _tf_load(path, label):
        img, label = tf.py_function(_load, [path, label], [tf.float32, tf.float32])
        img.set_shape(IMG_SIZE + (3,))
        label.set_shape([])
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        ds = ds.shuffle(buffer_size=len(dframe), seed=SEED)
    ds = ds.map(_tf_load, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (augment_layer(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = make_dataset(train_df, training=True)
val_ds   = make_dataset(val_df, training=False)
test_ds  = make_dataset(test_df, training=False)


In [ ]:

# Visualize a batch of augmented training images
images, labels = next(iter(train_ds))
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    if i < len(images):
        ax.imshow(images[i].numpy().astype("uint8"))
        lbl = "jaundice" if labels[i].numpy()==1 else "normal"
        ax.set_title(lbl, fontsize=11)
    ax.axis("off")
plt.suptitle("Augmented training batch")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/augmented_batch.png", dpi=150)
plt.show()


## 8. Class weights (handles the ~200 vs ~1100 imbalance)

In [ ]:

train_labels = train_df["label"].values
class_weights_arr = compute_class_weight("balanced", classes=np.unique(train_labels), y=train_labels)
class_weight_dict = {int(c): float(w) for c, w in zip(np.unique(train_labels), class_weights_arr)}
print("Class weights:", class_weight_dict)



## 9. CNN model — MobileNetV3-Small transfer learning

Matches the deck's Layer 2 "Risk Model: MobileNetV3-Small / EfficientNet-Lite"
box. Frozen ImageNet backbone + small trainable head first, then a short
fine-tune of the top backbone layers at a much lower learning rate.


In [ ]:

def build_model():
    base = keras.applications.MobileNetV3Small(
        input_shape=IMG_SIZE + (3,), include_top=False, weights="imagenet",
        include_preprocessing=True,  # handles its own scaling from 0-255 input
    )
    base.trainable = False

    inputs = keras.Input(shape=IMG_SIZE + (3,))
    x = base(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs)
    return model, base

model, base_model = build_model()

metrics = [
    keras.metrics.BinaryAccuracy(name="accuracy"),
    keras.metrics.AUC(name="auc"),
    keras.metrics.Precision(name="precision"),
    keras.metrics.Recall(name="recall"),
]
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=metrics)
model.summary()


## 10. Train — Phase 1 (head only), then Phase 2 (fine-tune)

In [ ]:

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=2),
    keras.callbacks.ModelCheckpoint(f"{OUT_DIR}/best_head.h5", monitor="val_auc", mode="max", save_best_only=True),
]

history1 = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
    class_weight=class_weight_dict, callbacks=callbacks,
)


In [ ]:

base_model.trainable = True
for layer in base_model.layers[:-20]:   # keep most of the backbone frozen
    layer.trainable = False

model.compile(optimizer=keras.optimizers.Adam(1e-5), loss="binary_crossentropy", metrics=metrics)

callbacks_ft = [
    keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_auc", mode="max", factor=0.5, patience=2),
    keras.callbacks.ModelCheckpoint(f"{OUT_DIR}/best_finetuned.h5", monitor="val_auc", mode="max", save_best_only=True),
]

history2 = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS_FINETUNE,
    class_weight=class_weight_dict, callbacks=callbacks_ft,
)


## 11. Training curves

In [ ]:

def combine_history(h1, h2, key):
    return h1.history.get(key, []) + h2.history.get(key, [])

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
plot_keys = [("loss","val_loss","Loss"), ("accuracy","val_accuracy","Accuracy"),
             ("auc","val_auc","AUC"), ("recall","val_recall","Recall")]
for ax, (k, vk, title) in zip(axes.flat, plot_keys):
    train_vals = combine_history(history1, history2, k)
    val_vals = combine_history(history1, history2, vk)
    ax.plot(train_vals, label="train")
    ax.plot(val_vals, label="val")
    ax.axvline(len(history1.history.get(k, [])) - 0.5, color="gray", linestyle="--", alpha=0.5, label="fine-tune start")
    ax.set_title(title); ax.set_xlabel("epoch"); ax.legend()
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/training_curves.png", dpi=150)
plt.show()


## 12. Evaluation on the held-out test set

In [ ]:

y_true, y_prob = [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0).ravel()
    y_prob.extend(preds.tolist())
    y_true.extend(labels.numpy().tolist())
y_true = np.array(y_true)
y_prob_cnn = np.array(y_prob)
y_pred_cnn = (y_prob_cnn >= 0.5).astype(int)

print("=== CNN (MobileNetV3) - default 0.5 threshold ===")
print(classification_report(y_true, y_pred_cnn, target_names=["normal","jaundice"]))
print("AUC-ROC:", roc_auc_score(y_true, y_prob_cnn))


In [ ]:

def plot_confusion(y_true, y_pred, title, path):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4.5,4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["normal","jaundice"], yticklabels=["normal","jaundice"])
    plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(title)
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()

plot_confusion(y_true, y_pred_cnn, "CNN Confusion Matrix (threshold=0.5)", f"{OUT_DIR}/cnn_confusion_matrix.png")


In [ ]:

fpr, tpr, _ = roc_curve(y_true, y_prob_cnn)
roc_auc_val = auc(fpr, tpr)

precisions, recalls, pr_thresholds = precision_recall_curve(y_true, y_prob_cnn)
ap = average_precision_score(y_true, y_prob_cnn)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].plot(fpr, tpr, label=f"AUC = {roc_auc_val:.3f}", color="#1C8C82")
axes[0].plot([0,1],[0,1], linestyle="--", color="gray")
axes[0].set_xlabel("False Positive Rate"); axes[0].set_ylabel("True Positive Rate")
axes[0].set_title("ROC Curve"); axes[0].legend()

axes[1].plot(recalls, precisions, label=f"AP = {ap:.3f}", color="#B23A2E")
axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall Curve"); axes[1].legend()

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/cnn_roc_pr_curves.png", dpi=150)
plt.show()


In [ ]:

# Recall-first threshold sweep - a missed jaundiced newborn is worse than a
# false alarm, so we look for the highest threshold that still hits a
# target recall, rather than defaulting to 0.5 / optimizing accuracy.
TARGET_RECALL = 0.90

thresholds = np.linspace(0.02, 0.98, 49)
rows = []
for t in thresholds:
    pred_t = (y_prob_cnn >= t).astype(int)
    rows.append({
        "threshold": t,
        "precision": precision_score(y_true, pred_t, zero_division=0),
        "recall": recall_score(y_true, pred_t, zero_division=0),
        "f1": f1_score(y_true, pred_t, zero_division=0),
    })
sweep_df = pd.DataFrame(rows)

plt.figure(figsize=(7,4.5))
plt.plot(sweep_df["threshold"], sweep_df["precision"], label="precision")
plt.plot(sweep_df["threshold"], sweep_df["recall"], label="recall")
plt.plot(sweep_df["threshold"], sweep_df["f1"], label="F1")
plt.axhline(TARGET_RECALL, color="gray", linestyle="--", alpha=0.6, label=f"target recall={TARGET_RECALL}")
plt.xlabel("decision threshold"); plt.ylabel("score"); plt.legend()
plt.title("Threshold sweep (CNN)")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/cnn_threshold_sweep.png", dpi=150)
plt.show()

eligible = sweep_df[sweep_df["recall"] >= TARGET_RECALL]
best_threshold = eligible["threshold"].max() if len(eligible) else 0.5
print(f"Suggested REFER_URGENTLY threshold for >= {TARGET_RECALL:.0%} recall: {best_threshold:.2f}")

y_pred_tuned = (y_prob_cnn >= best_threshold).astype(int)
print(f"\n=== CNN metrics @ tuned threshold {best_threshold:.2f} ===")
print(classification_report(y_true, y_pred_tuned, target_names=["normal","jaundice"]))


In [ ]:

cnn_metrics_summary = {
    "threshold_0.5": {
        "accuracy": accuracy_score(y_true, y_pred_cnn),
        "precision": precision_score(y_true, y_pred_cnn, zero_division=0),
        "recall": recall_score(y_true, y_pred_cnn, zero_division=0),
        "f1": f1_score(y_true, y_pred_cnn, zero_division=0),
        "auc_roc": roc_auc_score(y_true, y_prob_cnn),
    },
    f"threshold_{best_threshold:.2f}_tuned": {
        "accuracy": accuracy_score(y_true, y_pred_tuned),
        "precision": precision_score(y_true, y_pred_tuned, zero_division=0),
        "recall": recall_score(y_true, y_pred_tuned, zero_division=0),
        "f1": f1_score(y_true, y_pred_tuned, zero_division=0),
        "auc_roc": roc_auc_score(y_true, y_prob_cnn),
    },
}
print(json.dumps(cnn_metrics_summary, indent=2))



## 13. Explainability — Grad-CAM

Matches the deck's Layer 2 "Explainability" box: shows *where* in the image
the model is basing its jaundice call on, so it isn't a pure black box.


In [ ]:

def make_gradcam_heatmap(img_array, model, base_model):
    # base_model (include_top=False) output IS the final conv feature map,
    # so we don't need to search for "the last conv layer" separately -
    # that search is fragile across TF versions on nested/branching models.
    dense_layer = model.layers[-1]  # final Dense(1, sigmoid) layer

    with tf.GradientTape() as tape:
        conv_out = base_model(img_array[np.newaxis, ...], training=False)
        tape.watch(conv_out)
        gap = tf.reduce_mean(conv_out, axis=(1, 2))
        logit = tf.matmul(gap, dense_layer.weights[0]) + dense_layer.weights[1]
        class_channel = logit[:, 0]

    grads = tape.gradient(class_channel, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_out[0]
    heatmap = conv_out @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def overlay_heatmap(img_uint8, heatmap, alpha=0.4):
    heatmap_resized = cv2.resize(heatmap, (img_uint8.shape[1], img_uint8.shape[0]))
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
    overlaid = (img_uint8 * (1 - alpha) + heatmap_color * alpha).astype("uint8")
    return overlaid


In [ ]:

# Show Grad-CAM for a few true-positive jaundice test predictions
test_paths = test_df["path"].values
tp_indices = np.where((y_true == 1) & (y_pred_cnn == 1))[0][:4]

if len(tp_indices) == 0:
    print("No confident true-positive jaundice predictions found to visualize - "
          "try lowering the confidence bar or check class balance.")
else:
    fig, axes = plt.subplots(2, len(tp_indices), figsize=(4*len(tp_indices), 8))
    for i, idx in enumerate(tp_indices):
        img = load_and_preprocess(test_paths[idx], calibrate=True).astype("float32")
        heatmap = make_gradcam_heatmap(img, model, base_model)
        overlay = overlay_heatmap(img.astype("uint8"), heatmap)

        axes[0, i].imshow(img.astype("uint8")); axes[0, i].axis("off")
        axes[0, i].set_title(f"input (p={y_prob_cnn[idx]:.2f})", fontsize=10)
        axes[1, i].imshow(overlay); axes[1, i].axis("off")
        axes[1, i].set_title("Grad-CAM", fontsize=10)
    plt.suptitle("Explainability: where the CNN is looking")
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/gradcam_examples.png", dpi=150)
    plt.show()


## 14. Export the CNN for deployment (.h5 + quantized .tflite)

In [ ]:

keras_path = f"{OUT_DIR}/risk_model_keras.h5"
model.save(keras_path)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()
tflite_path = f"{OUT_DIR}/risk_model.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

deployment_meta = {
    "class_order": {"0": "normal", "1": "jaundice"},
    "input_size": IMG_SIZE,
    "suggested_threshold": float(best_threshold),
    "cnn_metrics": cnn_metrics_summary,
}
with open(f"{OUT_DIR}/deployment_meta.json", "w") as f:
    json.dump(deployment_meta, f, indent=2)

print(f"Saved Keras model  -> {keras_path}")
print(f"Saved TFLite model -> {tflite_path}  ({len(tflite_model)/1024:.0f} KB)")
print(f"Saved metadata      -> {OUT_DIR}/deployment_meta.json")
print("\nDownload risk_model_keras.h5 (or the .tflite) from the Colab file "
      "browser and drop it into backend/model/ in the project - main.py "
      "auto-detects and loads it.")



## 15. Comparison model — XGBoost on hand-crafted colour features

This mirrors the diagram's alternative "Feature Extraction → XGBoost" path
and `backend/feature_extraction.py` / `backend/train_model.py` in the main
project. It's included so you can **compare** a fast, fully explainable
model against the CNN — worth showing judges as evidence you evaluated
trade-offs rather than defaulting to "bigger model = better."


In [ ]:

FEATURE_NAMES = [
    "L_mean","L_std","L_median","A_mean","A_std","A_median",
    "B_mean","B_std","B_median","H_mean","H_std","S_mean","S_std",
    "V_mean","V_std","yellow_ratio","R_over_B_mean",
]

def skin_mask(img_bgr):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, np.array([0,15,40], dtype=np.uint8), np.array([50,200,255], dtype=np.uint8))
    if mask.mean() < 5:
        mask = np.full(img_bgr.shape[:2], 255, dtype=np.uint8)
    return mask

def extract_color_features(img_bgr):
    mask_bool = skin_mask(img_bgr) > 0
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB).astype(np.float32)
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
    L,A,B = lab[...,0][mask_bool], lab[...,1][mask_bool], lab[...,2][mask_bool]
    H,S,V = hsv[...,0][mask_bool], hsv[...,1][mask_bool], hsv[...,2][mask_bool]
    if L.size == 0:
        L,A,B = lab[...,0].ravel(), lab[...,1].ravel(), lab[...,2].ravel()
        H,S,V = hsv[...,0].ravel(), hsv[...,1].ravel(), hsv[...,2].ravel()
    b_img = img_bgr[...,0][mask_bool].astype(np.float32)
    r_img = img_bgr[...,2][mask_bool].astype(np.float32)
    r_over_b = float(np.mean(r_img / np.clip(b_img,1,255)))
    yellow_ratio = float(np.mean(np.logical_and(H>=15, H<=45))) if H.size else 0.0
    return np.array([
        np.mean(L), np.std(L), np.median(L), np.mean(A), np.std(A), np.median(A),
        np.mean(B), np.std(B), np.median(B), np.mean(H), np.std(H),
        np.mean(S), np.std(S), np.mean(V), np.std(V), yellow_ratio, r_over_b,
    ], dtype=np.float32)

def build_feature_matrix(dframe):
    X, y = [], []
    for _, row in dframe.iterrows():
        img = cv2.imread(row["path"])
        if img is None:
            continue
        img = gray_world_white_balance(img)
        X.append(extract_color_features(img))
        y.append(row["label"])
    return np.array(X), np.array(y, dtype=int)

X_train, y_train = build_feature_matrix(train_df)
X_val, y_val = build_feature_matrix(val_df)
X_test, y_test = build_feature_matrix(test_df)
print("Feature matrix shapes:", X_train.shape, X_val.shape, X_test.shape)


In [ ]:

scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f"scale_pos_weight (for imbalance) = {scale_pos_weight:.2f}")

xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="auc", random_state=SEED,
)
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
y_pred_xgb = (y_prob_xgb >= 0.5).astype(int)

print("=== XGBoost (colour features) - default 0.5 threshold ===")
print(classification_report(y_test, y_pred_xgb, target_names=["normal","jaundice"]))
print("AUC-ROC:", roc_auc_score(y_test, y_prob_xgb))

plot_confusion(y_test, y_pred_xgb, "XGBoost Confusion Matrix", f"{OUT_DIR}/xgb_confusion_matrix.png")


In [ ]:

# Feature importance - the explainability story for the XGBoost path
importances = xgb_model.feature_importances_
order = np.argsort(importances)[::-1]
plt.figure(figsize=(7,5))
plt.barh([FEATURE_NAMES[i] for i in order][::-1], importances[order][::-1], color="#1C8C82")
plt.title("XGBoost feature importance")
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/xgb_feature_importance.png", dpi=150)
plt.show()


In [ ]:

xgb_metrics_summary = {
    "accuracy": accuracy_score(y_test, y_pred_xgb),
    "precision": precision_score(y_test, y_pred_xgb, zero_division=0),
    "recall": recall_score(y_test, y_pred_xgb, zero_division=0),
    "f1": f1_score(y_test, y_pred_xgb, zero_division=0),
    "auc_roc": roc_auc_score(y_test, y_prob_xgb),
}
print(json.dumps(xgb_metrics_summary, indent=2))

import joblib
joblib.dump(xgb_model, f"{OUT_DIR}/risk_model_xgb.joblib")
print(f"Saved XGBoost model -> {OUT_DIR}/risk_model_xgb.joblib")


## 16. Side-by-side model comparison

In [ ]:

comparison = pd.DataFrame({
    "CNN (MobileNetV3, tuned threshold)": cnn_metrics_summary[f"threshold_{best_threshold:.2f}_tuned"],
    "XGBoost (colour features)": xgb_metrics_summary,
}).T[["accuracy","precision","recall","f1","auc_roc"]]

display(comparison.round(3))

comparison.plot(kind="bar", figsize=(9,5), colormap="viridis")
plt.title("CNN vs XGBoost — test set metrics")
plt.ylabel("score"); plt.ylim(0,1.05)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(f"{OUT_DIR}/model_comparison.png", dpi=150)
plt.show()



## 17. Summary & next steps

- Both models and all figures are saved under `OUT_DIR` (`/content/neolume_out`
  by default) — download the whole folder before your Colab session ends.
- **Deployment:** copy `risk_model_keras.h5` (or `risk_model.tflite`) and/or
  `risk_model_xgb.joblib` into `backend/model/` in the main project.
  `backend/main.py` auto-detects whichever is present, in priority order
  sklearn/XGBoost joblib → Keras → rule-based fallback.
- **Before trusting either model clinically:** this is trained on a public
  dataset without calibration-card-normalized images or confirmed TSB
  ground truth — re-validate against your field pilot data before treating
  the output as anything beyond a screening triage signal (see the deck's
  own "Validation Gap" risk callout).
- **If recall is still lower than you want:** collect more jaundice-class
  images before adding more augmentation — augmentation multiplies what's
  there, it doesn't add new visual diversity indefinitely.
